# Qwen + BLIP2 Feature Adapter for BanglaCalamityMMD

This notebook is designed for Google Colab memory limits.

Instead of normal few-shot prompting, this notebook uses:

1. Qwen2-VL as a multimodal feature extractor.
2. BLIP2 as an image feature extractor.
3. Bangla text TF-IDF features.
4. A lightweight LinearSVC classifier.
5. Accuracy, macro precision, macro recall, and macro F1 evaluation.

Important: This method does not use `image_id` as a feature because filenames may leak the class label.

In [ ]:
# Install required libraries
!pip -q install transformers accelerate bitsandbytes qwen-vl-utils scikit-learn pandas numpy pillow tqdm scipy joblib

In [ ]:
import os
import gc
import re
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.sparse import hstack, csr_matrix, save_npz, load_npz
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Set dataset paths

Change these paths according to your Google Drive folder structure.

Expected CSV columns:

- `image_id`
- `context`
- `category`

Expected image folders:

- Train images
- Validation images
- Test images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/AI_Project/BanglaCalamityMMD"

TRAIN_CSV = os.path.join(DATASET_ROOT, "Disaster_train.csv")
VAL_CSV   = os.path.join(DATASET_ROOT, "Disaster_validation.csv")
TEST_CSV  = os.path.join(DATASET_ROOT, "Disaster_test.csv")

TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, "Train")
VAL_IMG_DIR   = os.path.join(DATASET_ROOT, "Validation")
TEST_IMG_DIR  = os.path.join(DATASET_ROOT, "Test")

RESULT_DIR = os.path.join(DATASET_ROOT, "feature_classifier_results")
os.makedirs(RESULT_DIR, exist_ok=True)

print("Result folder:", RESULT_DIR)

## 2. Fix Bangla text and normalize labels

Your sample text looks like mojibake:

`à¦à¦¡à¦¼à§‡...`

That usually means Bangla text was decoded incorrectly. This notebook tries to repair it before training.

In [ ]:
CLASSES = [
    "Earthquake",
    "Flood",
    "Landslides",
    "Wildfire",
    "Tropical Storm",
    "Drought",
    "Human Damage",
    "Non Disaster"
]

LABEL_FIX = {
    "Earthquake": "Earthquake",
    "Earthquakes": "Earthquake",

    "Flood": "Flood",
    "Floods": "Flood",

    "Landslide": "Landslides",
    "Landslides": "Landslides",

    "Wildfire": "Wildfire",
    "Wildfires": "Wildfire",

    "Tropical Storm": "Tropical Storm",
    "Tropical Storms": "Tropical Storm",

    "Drought": "Drought",
    "Droughts": "Drought",

    "Human Damage": "Human Damage",
    "Human-Damage": "Human Damage",

    "Non Disaster": "Non Disaster",
    "Non-Disaster": "Non Disaster",
    "Non-disaster": "Non Disaster",
}

def fix_mojibake_text(x):
    if pd.isna(x):
        return ""

    x = str(x)

    if "à¦" in x or "à§" in x:
        try:
            return x.encode("latin1").decode("utf-8")
        except Exception:
            return x

    return x

def clean_text(x):
    x = fix_mojibake_text(x)
    x = x.replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def get_image_path(image_id, image_dir):
    image_id = str(image_id).strip()
    base = os.path.join(image_dir, image_id)

    for ext in [".jpg", ".jpeg", ".png", ".webp", ".JPG", ".PNG", ""]:
        p = base + ext
        if os.path.exists(p):
            return p

    return base

def load_split(csv_path, image_dir):
    df = pd.read_csv(csv_path)

    required_cols = ["image_id", "context", "category"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in {csv_path}: {missing_cols}")

    df["context"] = df["context"].apply(clean_text)
    df["category"] = df["category"].astype(str).str.strip().map(LABEL_FIX)

    if df["category"].isna().sum() > 0:
        print("Unmapped labels:")
        print(df[df["category"].isna()].head(10))
        raise ValueError("Some labels could not be mapped. Update LABEL_FIX.")

    df["image_path"] = df["image_id"].apply(lambda x: get_image_path(x, image_dir))

    missing_images = (~df["image_path"].apply(os.path.exists)).sum()

    print("\n", os.path.basename(csv_path))
    print("Rows:", len(df))
    print("Missing images:", missing_images)
    print(df["category"].value_counts())

    if missing_images > 0:
        print("\nExamples of missing images:")
        print(df[~df["image_path"].apply(os.path.exists)][["image_id", "image_path"]].head())

    return df

train_df = load_split(TRAIN_CSV, TRAIN_IMG_DIR)
val_df   = load_split(VAL_CSV, VAL_IMG_DIR)
test_df  = load_split(TEST_CSV, TEST_IMG_DIR)

print("\nSample cleaned text:")
print(train_df[["image_id", "context", "category"]].head(3))

In [ ]:
def evaluate(y_true, y_pred, name):
    print("\n" + "="*80)
    print(name)
    print("="*80)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"Accuracy        : {acc*100:.2f}%")
    print(f"Macro Precision : {prec*100:.2f}%")
    print(f"Macro Recall    : {rec*100:.2f}%")
    print(f"Macro F1        : {f1*100:.2f}%")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=CLASSES, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)

    print("\nConfusion Matrix:")
    display(cm_df)

    plt.figure(figsize=(11, 8.5))
    sns.heatmap(
        cm_df,
        annot=True,
        fmt="d",
        cmap="YlOrRd",
        linewidths=0.7,
        linecolor="white",
        cbar=True,
        square=True,
        annot_kws={"size": 10, "weight": "bold"}
    )

    plt.title(name + " - Confusion Matrix", fontsize=15, weight="bold", pad=16)
    plt.ylabel("Actual Label", fontsize=12, weight="bold")
    plt.xlabel("Predicted Label", fontsize=12, weight="bold")
    plt.xticks(rotation=35, ha="right", fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()

    safe_name = (
        name.replace(" ", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace("|", "_")
    )

    try:
        fig_path = os.path.join(RESULT_DIR, f"{safe_name}_confusion_matrix_heatmap.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        print("Saved confusion matrix heatmap:", fig_path)
    except Exception as e:
        print("Could not save confusion matrix image:", e)

    plt.show()

    return {
        "accuracy": acc,
        "macro_precision": prec,
        "macro_recall": rec,
        "macro_f1": f1
    }


# Part A: Qwen2-VL Feature Adapter + Bangla Text Classifier

This section loads Qwen2-VL in 4-bit and extracts hidden-state embeddings.

The model is not asked to generate the disaster class. It only provides numerical features.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

torch.cuda.empty_cache()
gc.collect()

qwen_quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

QWEN_ID = "Qwen/Qwen2-VL-2B-Instruct"

qwen_processor = AutoProcessor.from_pretrained(QWEN_ID, trust_remote_code=True)

qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_ID,
    quantization_config=qwen_quant,
    device_map="auto",
    trust_remote_code=True
)

qwen_model.eval()

try:
    qwen_processor.image_processor.max_pixels = 224 * 224
    qwen_processor.image_processor.min_pixels = 112 * 112
except Exception:
    pass

print("Qwen2-VL loaded.")

In [ ]:
def qwen_get_embedding(image_path, text, max_text_chars=250):
    text = clean_text(text)[:max_text_chars]

    image = Image.open(image_path).convert("RGB")
    image.thumbnail((224, 224))

    prompt = (
        "Classify the disaster type from the image and Bangla context. "
        "Use visual and textual evidence. Context: " + text
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    chat_text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[chat_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = qwen_model(
            **inputs,
            output_hidden_states=True,
            return_dict=True
        )

        emb = outputs.hidden_states[-1][:, -1, :].detach().float().cpu().numpy()[0]

    del inputs, outputs
    torch.cuda.empty_cache()

    return emb

def build_qwen_embeddings(df, save_path):
    if os.path.exists(save_path):
        print("Loading saved embeddings:", save_path)
        return np.load(save_path)

    embeddings = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Qwen embeddings"):
        if not os.path.exists(row["image_path"]):
            raise FileNotFoundError(row["image_path"])

        emb = qwen_get_embedding(row["image_path"], row["context"])
        embeddings.append(emb)

    embeddings = np.vstack(embeddings).astype("float32")
    np.save(save_path, embeddings)
    print("Saved:", save_path)

    return embeddings

In [ ]:
qwen_train_path = os.path.join(RESULT_DIR, "qwen_train_embeddings.npy")
qwen_val_path   = os.path.join(RESULT_DIR, "qwen_val_embeddings.npy")
qwen_test_path  = os.path.join(RESULT_DIR, "qwen_test_embeddings.npy")

Xq_train = build_qwen_embeddings(train_df, qwen_train_path)
Xq_val   = build_qwen_embeddings(val_df, qwen_val_path)
Xq_test  = build_qwen_embeddings(test_df, qwen_test_path)

print("Qwen embedding shapes:")
print(Xq_train.shape, Xq_val.shape, Xq_test.shape)

In [ ]:
# Bangla text features
qwen_text_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    max_features=60000,
    min_df=2,
    sublinear_tf=True
)

Xt_train = qwen_text_vectorizer.fit_transform(train_df["context"])
Xt_val   = qwen_text_vectorizer.transform(val_df["context"])
Xt_test  = qwen_text_vectorizer.transform(test_df["context"])

# Combine Qwen features + Bangla text features
X_train = hstack([csr_matrix(Xq_train), Xt_train])
X_val   = hstack([csr_matrix(Xq_val), Xt_val])
X_test  = hstack([csr_matrix(Xq_test), Xt_test])

y_train = train_df["category"].values
y_val   = val_df["category"].values
y_test  = test_df["category"].values

print("Final Qwen feature shapes:")
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
best_qwen_model = None
best_qwen_f1 = -1
best_qwen_c = None

for c in [0.05, 0.1, 0.3, 0.5, 1, 2, 3, 5, 8]:
    clf = LinearSVC(
        C=c,
        class_weight="balanced",
        max_iter=10000,
        random_state=SEED
    )

    clf.fit(X_train, y_train)
    val_pred = clf.predict(X_val)

    val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
    val_acc = accuracy_score(y_val, val_pred)

    print(f"C={c:<4} | Val Accuracy={val_acc*100:.2f}% | Val Macro F1={val_f1*100:.2f}%")

    if val_f1 > best_qwen_f1:
        best_qwen_f1 = val_f1
        best_qwen_c = c
        best_qwen_model = clf

print("\nBest Qwen C:", best_qwen_c)

In [ ]:
qwen_test_pred = best_qwen_model.predict(X_test)

qwen_metrics = evaluate(
    y_true=y_test,
    y_pred=qwen_test_pred,
    name="Qwen2-VL Feature Extractor + Bangla Text Classifier"
)

qwen_out = test_df[["image_id", "context", "category", "image_path"]].copy()
qwen_out["prediction"] = qwen_test_pred
qwen_out.to_csv(os.path.join(RESULT_DIR, "qwen_feature_classifier_predictions.csv"), index=False)

joblib.dump(best_qwen_model, os.path.join(RESULT_DIR, "qwen_linearsvc.joblib"))
joblib.dump(qwen_text_vectorizer, os.path.join(RESULT_DIR, "qwen_text_vectorizer.joblib"))

print("Saved Qwen predictions and classifier.")

## Clear Qwen from GPU memory before loading BLIP2

Run this before the BLIP2 section. Colab RAM is not a magical bottomless pit, sadly.

In [ ]:
del qwen_model
del qwen_processor
torch.cuda.empty_cache()
gc.collect()

print("Qwen removed from GPU memory.")

# Part B: BLIP2 Image Feature Adapter + Bangla Text Classifier

BLIP2 is not strong for Bangla text generation. This notebook uses BLIP2 only for image embeddings and uses TF-IDF for the Bangla text side.

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

torch.cuda.empty_cache()
gc.collect()

blip_quant = BitsAndBytesConfig(load_in_8bit=True)

BLIP_ID = "Salesforce/blip2-opt-2.7b"

blip_processor = Blip2Processor.from_pretrained(BLIP_ID)

blip_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP_ID,
    quantization_config=blip_quant,
    device_map="auto",
    torch_dtype=torch.float16
)

blip_model.eval()

print("BLIP2 loaded.")

In [ ]:
def blip2_get_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    image.thumbnail((224, 224))

    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    ).to("cuda", torch.float16)

    with torch.no_grad():
        outputs = blip_model.vision_model(
            pixel_values=inputs["pixel_values"],
            return_dict=True
        )

        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            emb = outputs.pooler_output.detach().float().cpu().numpy()[0]
        else:
            emb = outputs.last_hidden_state.mean(dim=1).detach().float().cpu().numpy()[0]

    del inputs, outputs
    torch.cuda.empty_cache()

    return emb

def build_blip2_embeddings(df, save_path):
    if os.path.exists(save_path):
        print("Loading saved embeddings:", save_path)
        return np.load(save_path)

    embeddings = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="BLIP2 image embeddings"):
        if not os.path.exists(row["image_path"]):
            raise FileNotFoundError(row["image_path"])

        emb = blip2_get_image_embedding(row["image_path"])
        embeddings.append(emb)

    embeddings = np.vstack(embeddings).astype("float32")
    np.save(save_path, embeddings)
    print("Saved:", save_path)

    return embeddings

In [ ]:
blip_train_path = os.path.join(RESULT_DIR, "blip2_train_embeddings.npy")
blip_val_path   = os.path.join(RESULT_DIR, "blip2_val_embeddings.npy")
blip_test_path  = os.path.join(RESULT_DIR, "blip2_test_embeddings.npy")

Xb_train = build_blip2_embeddings(train_df, blip_train_path)
Xb_val   = build_blip2_embeddings(val_df, blip_val_path)
Xb_test  = build_blip2_embeddings(test_df, blip_test_path)

print("BLIP2 embedding shapes:")
print(Xb_train.shape, Xb_val.shape, Xb_test.shape)

In [ ]:
blip_text_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    max_features=60000,
    min_df=2,
    sublinear_tf=True
)

Xt_train_b = blip_text_vectorizer.fit_transform(train_df["context"])
Xt_val_b   = blip_text_vectorizer.transform(val_df["context"])
Xt_test_b  = blip_text_vectorizer.transform(test_df["context"])

X_train_b = hstack([csr_matrix(Xb_train), Xt_train_b])
X_val_b   = hstack([csr_matrix(Xb_val), Xt_val_b])
X_test_b  = hstack([csr_matrix(Xb_test), Xt_test_b])

y_train = train_df["category"].values
y_val   = val_df["category"].values
y_test  = test_df["category"].values

print("Final BLIP2 feature shapes:")
print(X_train_b.shape, X_val_b.shape, X_test_b.shape)

In [ ]:
best_blip_model = None
best_blip_f1 = -1
best_blip_c = None

for c in [0.05, 0.1, 0.3, 0.5, 1, 2, 3, 5, 8]:
    clf = LinearSVC(
        C=c,
        class_weight="balanced",
        max_iter=10000,
        random_state=SEED
    )

    clf.fit(X_train_b, y_train)
    val_pred = clf.predict(X_val_b)

    val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
    val_acc = accuracy_score(y_val, val_pred)

    print(f"C={c:<4} | Val Accuracy={val_acc*100:.2f}% | Val Macro F1={val_f1*100:.2f}%")

    if val_f1 > best_blip_f1:
        best_blip_f1 = val_f1
        best_blip_c = c
        best_blip_model = clf

print("\nBest BLIP2 C:", best_blip_c)

In [ ]:
blip_test_pred = best_blip_model.predict(X_test_b)

blip_metrics = evaluate(
    y_true=y_test,
    y_pred=blip_test_pred,
    name="BLIP2 Image Feature Extractor + Bangla Text Classifier"
)

blip_out = test_df[["image_id", "context", "category", "image_path"]].copy()
blip_out["prediction"] = blip_test_pred
blip_out.to_csv(os.path.join(RESULT_DIR, "blip2_feature_classifier_predictions.csv"), index=False)

joblib.dump(best_blip_model, os.path.join(RESULT_DIR, "blip2_linearsvc.joblib"))
joblib.dump(blip_text_vectorizer, os.path.join(RESULT_DIR, "blip2_text_vectorizer.joblib"))

print("Saved BLIP2 predictions and classifier.")

# Optional: Compare Qwen and BLIP2 Results

This cell prints the final summary if both sections were completed in the same runtime.

In [ ]:
summary_rows = []

try:
    summary_rows.append({
        "model": "Qwen2-VL features + Bangla TF-IDF + LinearSVC",
        **qwen_metrics
    })
except NameError:
    pass

try:
    summary_rows.append({
        "model": "BLIP2 image features + Bangla TF-IDF + LinearSVC",
        **blip_metrics
    })
except NameError:
    pass

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
    summary_df.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"), index=False)
else:
    print("No metrics found yet. Run Qwen and/or BLIP2 sections first.")

# Notes

## Why this notebook avoids normal few-shot generation

Normal few-shot generation gives the VLM a few labeled examples and asks it to output a class name. That is expensive, unstable, and memory-heavy because every image and example becomes part of the prompt.

This notebook does something different:

- It extracts compact numerical features from Qwen or BLIP2.
- It combines those features with Bangla text features.
- It trains a small classifier on the actual training set.
- It avoids increasing `k`, which causes Colab out-of-memory errors.
- It avoids using `image_id`, because that can leak the answer.

This approach is more suitable when you have a proper train, validation, and test split.